# Análisis de Evaluación de Sistemas de Recomendación

Este notebook analiza los resultados de la evaluación offline de diferentes estrategias de recomendación usando múltiples métricas.

## Métricas evaluadas:
- **NDCG@9**: Normalized Discounted Cumulative Gain
- **Precision@9**: Proporción de recomendaciones relevantes
- **Recall@9**: Proporción de items relevantes recuperados
- **F1@9**: Media armónica de Precision y Recall
- **MRR**: Mean Reciprocal Rank (posición del primer item relevante)
- **Genre Diversity**: Diversidad de géneros en las recomendaciones
- **Artist Diversity**: Diversidad de artistas en las recomendaciones
- **Novelty**: Novedad basada en popularidad (-log2)

## Estrategias evaluadas:
- **hybrid**: Sistema híbrido completo
- **advanced**: Recomendaciones avanzadas (NMF + Two Towers)
- **nmf**: Factorización matricial (NMF)
- **two_towers**: Two Towers (Deep Learning)
- **pairs**: Co-ocurrencia (release_pairs)
- **content**: Perfiles de contenido
- **random**: Exploración aleatoria (baseline)
- **popular**: Popularidad (baseline)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


# Configurar estilo
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["font.size"] = 11

# Cargar datos
csv_path = Path("../offline_recommender/output/resultados_evaluacion_completa.csv")
df = pd.read_csv(csv_path)

print(f"📊 Datos cargados: {len(df)} usuarios evaluados")
print(f"📈 Columnas: {len(df.columns)}")
print("\nPrimeras filas:")
df.head()

In [ ]:
# Definir estrategias y métricas
strategies = ["hybrid", "advanced", "nmf", "two_towers", "pairs", "content", "random", "popular"]
metrics = [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]

# Calcular estadísticas descriptivas por estrategia y métrica
print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS POR ESTRATEGIA")
print("=" * 80)

for strategy in strategies:
    print(f"\n{strategy.upper()}:")
    print("-" * 40)
    for metric in metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            values = df[col].dropna()
            if len(values) > 0:
                print(
                    f"  {metric:20s}: mean={values.mean():.4f}  std={values.std():.4f}  min={values.min():.4f}  max={values.max():.4f}"
                )

In [ ]:
# Crear DataFrame con promedios por estrategia
summary_data = []
for strategy in strategies:
    row = {"strategy": strategy}
    for metric in metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            row[metric] = df[col].mean()
        else:
            row[metric] = 0.0
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df.set_index("strategy")

print("=" * 80)
print("PROMEDIOS POR ESTRATEGIA")
print("=" * 80)
print(summary_df.round(4))

In [ ]:
# Visualización 1: Comparación de métricas principales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Comparación de Estrategias por Métricas Principales", fontsize=16, fontweight="bold")

# NDCG
ax1 = axes[0, 0]
summary_df["ndcg"].plot(kind="bar", ax=ax1, color="steelblue")
ax1.set_title("NDCG@9", fontweight="bold")
ax1.set_ylabel("NDCG")
ax1.set_xlabel("Estrategia")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# Precision
ax2 = axes[0, 1]
summary_df["precision"].plot(kind="bar", ax=ax2, color="coral")
ax2.set_title("Precision@9", fontweight="bold")
ax2.set_ylabel("Precision")
ax2.set_xlabel("Estrategia")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

# Recall
ax3 = axes[1, 0]
summary_df["recall"].plot(kind="bar", ax=ax3, color="mediumseagreen")
ax3.set_title("Recall@9", fontweight="bold")
ax3.set_ylabel("Recall")
ax3.set_xlabel("Estrategia")
ax3.tick_params(axis="x", rotation=45)
ax3.grid(axis="y", alpha=0.3)

# F1
ax4 = axes[1, 1]
summary_df["f1"].plot(kind="bar", ax=ax4, color="mediumpurple")
ax4.set_title("F1@9", fontweight="bold")
ax4.set_ylabel("F1 Score")
ax4.set_xlabel("Estrategia")
ax4.tick_params(axis="x", rotation=45)
ax4.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 2: Heatmap de todas las métricas
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    summary_df.T,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    cbar_kws={"label": "Valor promedio"},
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Heatmap de Métricas por Estrategia", fontsize=14, fontweight="bold", pad=20)
ax.set_xlabel("Estrategia", fontweight="bold")
ax.set_ylabel("Métrica", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 3: Diversidad y Novedad
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Análisis de Diversidad y Novedad", fontsize=16, fontweight="bold")

# Genre Diversity
ax1 = axes[0]
summary_df["genre_diversity"].plot(kind="bar", ax=ax1, color="teal")
ax1.set_title("Diversidad de Géneros", fontweight="bold")
ax1.set_ylabel("Géneros únicos / Releases")
ax1.set_xlabel("Estrategia")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# Artist Diversity
ax2 = axes[1]
summary_df["artist_diversity"].plot(kind="bar", ax=ax2, color="orange")
ax2.set_title("Diversidad de Artistas", fontweight="bold")
ax2.set_ylabel("Artistas únicos / Releases")
ax2.set_xlabel("Estrategia")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

# Novelty
ax3 = axes[2]
summary_df["novelty"].plot(kind="bar", ax=ax3, color="crimson")
ax3.set_title("Novedad", fontweight="bold")
ax3.set_ylabel("Novedad promedio (-log2)")
ax3.set_xlabel("Estrategia")
ax3.tick_params(axis="x", rotation=45)
ax3.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Visualización 4: Boxplots de distribución de métricas principales
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Distribución de Métricas por Estrategia", fontsize=16, fontweight="bold")

# Preparar datos para boxplots
ndcg_data = [df[f"{s}_ndcg"].dropna() for s in strategies]
precision_data = [df[f"{s}_precision"].dropna() for s in strategies]
recall_data = [df[f"{s}_recall"].dropna() for s in strategies]
f1_data = [df[f"{s}_f1"].dropna() for s in strategies]

# NDCG
axes[0, 0].boxplot(ndcg_data, labels=strategies)
axes[0, 0].set_title("NDCG@9", fontweight="bold")
axes[0, 0].set_ylabel("NDCG")
axes[0, 0].tick_params(axis="x", rotation=45)
axes[0, 0].grid(axis="y", alpha=0.3)

# Precision
axes[0, 1].boxplot(precision_data, labels=strategies)
axes[0, 1].set_title("Precision@9", fontweight="bold")
axes[0, 1].set_ylabel("Precision")
axes[0, 1].tick_params(axis="x", rotation=45)
axes[0, 1].grid(axis="y", alpha=0.3)

# Recall
axes[1, 0].boxplot(recall_data, labels=strategies)
axes[1, 0].set_title("Recall@9", fontweight="bold")
axes[1, 0].set_ylabel("Recall")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].grid(axis="y", alpha=0.3)

# F1
axes[1, 1].boxplot(f1_data, labels=strategies)
axes[1, 1].set_title("F1@9", fontweight="bold")
axes[1, 1].set_ylabel("F1 Score")
axes[1, 1].tick_params(axis="x", rotation=45)
axes[1, 1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Ranking de estrategias por métrica
print("=" * 80)
print("RANKING DE ESTRATEGIAS POR MÉTRICA")
print("=" * 80)

for metric in [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]:
    print(f"\n{metric.upper()}:")
    print("-" * 40)
    ranked = summary_df[metric].sort_values(ascending=False)
    for i, (strategy, value) in enumerate(ranked.items(), 1):
        print(f"  {i}. {strategy:15s}: {value:.4f}")

In [ ]:
# Análisis de correlación entre métricas (para estrategias principales)
correlation_metrics = [
    "ndcg",
    "precision",
    "recall",
    "f1",
    "mrr",
    "genre_diversity",
    "artist_diversity",
    "novelty",
]
corr_data = []

for strategy in ["hybrid", "advanced", "pairs", "content"]:
    row = {}
    for metric in correlation_metrics:
        col = f"{strategy}_{metric}"
        if col in df.columns:
            row[metric] = df[col].mean()
    corr_data.append(row)

corr_df = pd.DataFrame(corr_data, index=["hybrid", "advanced", "pairs", "content"])

fig, ax = plt.subplots(figsize=(10, 8))
correlation_matrix = corr_df.corr()
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación"},
    ax=ax,
)
ax.set_title(
    "Correlación entre Métricas (Estrategias principales)", fontsize=14, fontweight="bold", pad=20
)
plt.tight_layout()
plt.show()

In [ ]:
# Análisis por tamaño de holdout
df["holdout_category"] = pd.cut(
    df["holdout_size"],
    bins=[0, 20, 50, 100, 200, float("inf")],
    labels=[
        "Muy pequeño (≤20)",
        "Pequeño (21-50)",
        "Mediano (51-100)",
        "Grande (101-200)",
        "Muy grande (>200)",
    ],
)

print("=" * 80)
print("ANÁLISIS POR TAMAÑO DE HOLDOUT")
print("=" * 80)

for category in df["holdout_category"].cat.categories:
    subset = df[df["holdout_category"] == category]
    if len(subset) > 0:
        print(f"\n{category} ({len(subset)} usuarios):")
        print("-" * 40)
        for strategy in ["hybrid", "advanced", "pairs", "content"]:
            ndcg_col = f"{strategy}_ndcg"
            if ndcg_col in subset.columns:
                mean_ndcg = subset[ndcg_col].mean()
                print(f"  {strategy:15s}: NDCG promedio = {mean_ndcg:.4f}")

In [ ]:
# Visualización 5: Trade-off entre relevancia y diversidad
fig, ax = plt.subplots(figsize=(12, 8))

for strategy in strategies:
    ndcg_col = f"{strategy}_ndcg"
    div_col = f"{strategy}_genre_diversity"

    if ndcg_col in df.columns and div_col in df.columns:
        mean_ndcg = df[ndcg_col].mean()
        mean_div = df[div_col].mean()
        ax.scatter(mean_ndcg, mean_div, s=200, alpha=0.7, label=strategy)
        ax.annotate(
            strategy, (mean_ndcg, mean_div), xytext=(5, 5), textcoords="offset points", fontsize=9
        )

ax.set_xlabel("NDCG@9 (Relevancia)", fontweight="bold", fontsize=12)
ax.set_ylabel("Diversidad de Géneros", fontweight="bold", fontsize=12)
ax.set_title("Trade-off: Relevancia vs Diversidad", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="best", framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visualización 6: Trade-off entre relevancia y novedad
fig, ax = plt.subplots(figsize=(12, 8))

for strategy in strategies:
    ndcg_col = f"{strategy}_ndcg"
    nov_col = f"{strategy}_novelty"

    if ndcg_col in df.columns and nov_col in df.columns:
        mean_ndcg = df[ndcg_col].mean()
        mean_nov = df[nov_col].mean()
        ax.scatter(mean_ndcg, mean_nov, s=200, alpha=0.7, label=strategy)
        ax.annotate(
            strategy, (mean_ndcg, mean_nov), xytext=(5, 5), textcoords="offset points", fontsize=9
        )

ax.set_xlabel("NDCG@9 (Relevancia)", fontweight="bold", fontsize=12)
ax.set_ylabel("Novedad", fontweight="bold", fontsize=12)
ax.set_title("Trade-off: Relevancia vs Novedad", fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="best", framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusiones y Recomendaciones

### Resumen de Hallazgos:

1. **Mejor relevancia (NDCG)**: 
   - Estrategias con mejor NDCG promedio

2. **Mejor precisión**: 
   - Estrategias que mejor recuperan items relevantes

3. **Mejor diversidad**: 
   - Estrategias que ofrecen más variedad en géneros/artistas

4. **Mejor novedad**: 
   - Estrategias que descubren items menos populares

5. **Trade-offs identificados**:
   - Relación entre relevancia y diversidad
   - Relación entre relevancia y novedad

### Recomendaciones:

- **Para usuarios con historial corto**: 
- **Para usuarios con historial largo**: 
- **Para maximizar descubrimiento**: 
- **Para maximizar relevancia**:


In [ ]:
# Guardar resumen en CSV
summary_df.to_csv("../resultados_evaluacion_resumen.csv")
print("✅ Resumen guardado en: resultados_evaluacion_resumen.csv")

# Estadísticas adicionales
print("\n" + "=" * 80)
print("ESTADÍSTICAS ADICIONALES")
print("=" * 80)
print(f"\nTotal de usuarios evaluados: {len(df)}")
print(f"Tamaño promedio de holdout: {df['holdout_size'].mean():.1f}")
print(f"Tamaño mínimo de holdout: {df['holdout_size'].min()}")
print(f"Tamaño máximo de holdout: {df['holdout_size'].max()}")

# Usuarios con mejor rendimiento por estrategia
print("\n" + "=" * 80)
print("TOP 5 USUARIOS POR ESTRATEGIA (NDCG)")
print("=" * 80)
for strategy in ["hybrid", "advanced", "pairs", "content"]:
    ndcg_col = f"{strategy}_ndcg"
    if ndcg_col in df.columns:
        top_users = df.nlargest(5, ndcg_col)[["user_id", "holdout_size", ndcg_col]]
        print(f"\n{strategy.upper()}:")
        print(top_users.to_string(index=False))

## Análisis Avanzado para Sistema Híbrido

Los siguientes análisis están diseñados para obtener insights que permitan construir un sistema híbrido más poderoso, ya sea mediante:
- **Modelado adaptativo**: Seleccionar la mejor estrategia según características del usuario
- **Ensamble inteligente**: Combinar múltiples estrategias con pesos adaptativos
- **Detección de casos especiales**: Identificar usuarios que requieren tratamiento especial


In [ ]:
# Análisis 1: Complementariedad entre estrategias
# Identificar cuándo una estrategia funciona mejor que otra para el mismo usuario

print("=" * 80)
print("ANÁLISIS DE COMPLEMENTARIEDAD ENTRE ESTRATEGIAS")
print("=" * 80)

# Comparar estrategias principales
main_strategies = ["pairs", "content", "advanced", "hybrid"]
comparison_results = []

for i, strat1 in enumerate(main_strategies):
    for strat2 in main_strategies[i + 1 :]:
        col1 = f"{strat1}_ndcg"
        col2 = f"{strat2}_ndcg"

        if col1 in df.columns and col2 in df.columns:
            # Usuarios donde strat1 es mejor
            strat1_better = df[df[col1] > df[col2]]
            # Usuarios donde strat2 es mejor
            strat2_better = df[df[col2] > df[col1]]
            # Usuarios donde ambas son iguales (o ambas malas)
            both_equal = df[(df[col1] == df[col2]) | ((df[col1] == 0) & (df[col2] == 0))]

            comparison_results.append(
                {
                    "comparison": f"{strat1} vs {strat2}",
                    f"{strat1}_better_count": len(strat1_better),
                    f"{strat1}_better_mean_diff": (strat1_better[col1] - strat1_better[col2]).mean()
                    if len(strat1_better) > 0
                    else 0,
                    f"{strat2}_better_count": len(strat2_better),
                    f"{strat2}_better_mean_diff": (strat2_better[col2] - strat2_better[col1]).mean()
                    if len(strat2_better) > 0
                    else 0,
                    "both_equal_count": len(both_equal),
                    "total_users": len(df),
                }
            )

comparison_df = pd.DataFrame(comparison_results)
print("\nComparaciones de rendimiento:")
print(comparison_df.to_string(index=False))

# Identificar usuarios donde diferentes estrategias funcionan mejor
df["best_strategy"] = df[[f"{s}_ndcg" for s in main_strategies]].idxmax(axis=1)
df["best_strategy"] = df["best_strategy"].str.replace("_ndcg", "")

print("\n" + "=" * 80)
print("DISTRIBUCIÓN DE MEJOR ESTRATEGIA POR USUARIO")
print("=" * 80)
print(df["best_strategy"].value_counts())
print("\nPorcentaje de usuarios donde cada estrategia es la mejor:")
print((df["best_strategy"].value_counts() / len(df) * 100).round(2))

In [ ]:
# Análisis 2: Características de usuarios que predicen mejor rendimiento
# Relacionar tamaño de holdout y otras características con rendimiento de cada estrategia

print("=" * 80)
print("ANÁLISIS DE CARACTERÍSTICAS DE USUARIOS VS RENDIMIENTO")
print("=" * 80)

# Crear segmentos de usuarios por tamaño de holdout
df["holdout_segment"] = pd.cut(
    df["holdout_size"],
    bins=[0, 10, 20, 50, 100, 200, float("inf")],
    labels=[
        "Muy pequeño (≤10)",
        "Pequeño (11-20)",
        "Mediano (21-50)",
        "Grande (51-100)",
        "Muy grande (101-200)",
        "Enorme (>200)",
    ],
)

# Análisis por segmento
segment_analysis = []
for segment in df["holdout_segment"].cat.categories:
    subset = df[df["holdout_segment"] == segment]
    if len(subset) > 0:
        row = {"segment": segment, "user_count": len(subset)}
        for strategy in main_strategies:
            col = f"{strategy}_ndcg"
            if col in subset.columns:
                row[f"{strategy}_mean"] = subset[col].mean()
                row[f"{strategy}_median"] = subset[col].median()
                row[f"{strategy}_std"] = subset[col].std()
        segment_analysis.append(row)

segment_df = pd.DataFrame(segment_analysis)
print("\nRendimiento promedio por segmento de holdout:")
print(segment_df.round(4).to_string(index=False))

# Visualización: Rendimiento por tamaño de holdout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Rendimiento de Estrategias por Tamaño de Holdout", fontsize=16, fontweight="bold")

for idx, strategy in enumerate(main_strategies):
    ax = axes[idx // 2, idx % 2]
    col = f"{strategy}_ndcg"

    if col in df.columns:
        # Boxplot por segmento
        data_by_segment = [
            df[df["holdout_segment"] == seg][col].dropna()
            for seg in df["holdout_segment"].cat.categories
        ]

        bp = ax.boxplot(
            data_by_segment, labels=df["holdout_segment"].cat.categories, patch_artist=True
        )
        for patch in bp["boxes"]:
            patch.set_facecolor("lightblue")

        ax.set_title(f"{strategy.upper()}", fontweight="bold")
        ax.set_ylabel("NDCG@9")
        ax.set_xlabel("Tamaño de Holdout")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análisis 3: Potencial de Ensamble
# Identificar usuarios donde múltiples estrategias funcionan bien simultáneamente

print("=" * 80)
print("ANÁLISIS DE POTENCIAL DE ENSAMBLE")
print("=" * 80)

# Definir umbral de "buen rendimiento" (percentil 75)
thresholds = {}
for strategy in main_strategies:
    col = f"{strategy}_ndcg"
    if col in df.columns:
        thresholds[strategy] = df[col].quantile(0.75)

print("\nUmbrales de buen rendimiento (percentil 75):")
for strat, thresh in thresholds.items():
    print(f"  {strat:15s}: {thresh:.4f}")

# Identificar usuarios donde múltiples estrategias funcionan bien
df["good_strategies_count"] = 0
for strategy in main_strategies:
    col = f"{strategy}_ndcg"
    if col in df.columns:
        df["good_strategies_count"] += (df[col] >= thresholds[strategy]).astype(int)

print("\n" + "=" * 80)
print("USUARIOS CON MÚLTIPLES ESTRATEGIAS FUNCIONANDO BIEN")
print("=" * 80)
print(df["good_strategies_count"].value_counts().sort_index())
print(
    f"\nUsuarios con 2+ estrategias funcionando bien: {(df['good_strategies_count'] >= 2).sum()} ({(df['good_strategies_count'] >= 2).sum() / len(df) * 100:.1f}%)"
)
print(
    f"Usuarios con 3+ estrategias funcionando bien: {(df['good_strategies_count'] >= 3).sum()} ({(df['good_strategies_count'] >= 3).sum() / len(df) * 100:.1f}%)"
)

# Análisis de combinaciones más comunes
print("\n" + "=" * 80)
print("COMBINACIONES MÁS COMUNES DE ESTRATEGIAS QUE FUNCIONAN BIEN")
print("=" * 80)

# Crear columnas binarias para cada estrategia
for strategy in main_strategies:
    col = f"{strategy}_ndcg"
    if col in df.columns:
        df[f"{strategy}_good"] = (df[col] >= thresholds[strategy]).astype(int)

# Encontrar combinaciones más comunes
from itertools import combinations


combination_counts = {}

for r in range(2, len(main_strategies) + 1):
    for combo in combinations(main_strategies, r):
        combo_cols = [f"{s}_good" for s in combo]
        if all(col in df.columns for col in combo_cols):
            count = (df[combo_cols].sum(axis=1) == len(combo)).sum()
            if count > 0:
                combination_counts[" + ".join(combo)] = count

if combination_counts:
    combo_df = pd.DataFrame(list(combination_counts.items()), columns=["Combinación", "Usuarios"])
    combo_df = combo_df.sort_values("Usuarios", ascending=False)
    print(combo_df.head(10).to_string(index=False))

In [ ]:
# Análisis 4: Casos donde el sistema híbrido falla
# Identificar usuarios donde ninguna estrategia funciona bien

print("=" * 80)
print("ANÁLISIS DE CASOS DONDE EL SISTEMA HÍBRIDO FALLA")
print("=" * 80)

# Usuarios donde todas las estrategias tienen bajo rendimiento
low_performance_threshold = df["hybrid_ndcg"].quantile(0.25)  # Percentil 25
df["all_strategies_poor"] = True

for strategy in main_strategies:
    col = f"{strategy}_ndcg"
    if col in df.columns:
        df["all_strategies_poor"] = df["all_strategies_poor"] & (
            df[col] < low_performance_threshold
        )

poor_performers = df[df["all_strategies_poor"]]
print(
    f"\nUsuarios con bajo rendimiento en todas las estrategias: {len(poor_performers)} ({len(poor_performers)/len(df)*100:.1f}%)"
)
print(f"Umbral usado (percentil 25 de hybrid_ndcg): {low_performance_threshold:.4f}")

if len(poor_performers) > 0:
    print("\nCaracterísticas de usuarios con bajo rendimiento:")
    print(f"  Tamaño promedio de holdout: {poor_performers['holdout_size'].mean():.1f}")
    print(f"  Tamaño mediano de holdout: {poor_performers['holdout_size'].median():.1f}")
    print(
        f"  Rango de holdout: [{poor_performers['holdout_size'].min()}, {poor_performers['holdout_size'].max()}]"
    )

    print("\nDistribución por segmento de holdout:")
    print(poor_performers["holdout_segment"].value_counts())

# Visualización: Comparación de usuarios con buen vs mal rendimiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análisis de Usuarios con Bajo Rendimiento", fontsize=14, fontweight="bold")

# Distribución de holdout_size
ax1 = axes[0]
ax1.hist(
    df[~df["all_strategies_poor"]]["holdout_size"],
    bins=50,
    alpha=0.6,
    label="Buen rendimiento",
    color="green",
)
ax1.hist(poor_performers["holdout_size"], bins=50, alpha=0.6, label="Bajo rendimiento", color="red")
ax1.set_xlabel("Tamaño de Holdout")
ax1.set_ylabel("Frecuencia")
ax1.set_title("Distribución de Tamaño de Holdout")
ax1.legend()
ax1.grid(alpha=0.3)

# Boxplot de NDCG por estrategia
ax2 = axes[1]
data_good = [df[~df["all_strategies_poor"]][f"{s}_ndcg"].dropna() for s in main_strategies]
data_poor = [poor_performers[f"{s}_ndcg"].dropna() for s in main_strategies]

x_pos = np.arange(len(main_strategies))
width = 0.35

means_good = [d.mean() if len(d) > 0 else 0 for d in data_good]
means_poor = [d.mean() if len(d) > 0 else 0 for d in data_poor]

ax2.bar(x_pos - width / 2, means_good, width, label="Buen rendimiento", color="green", alpha=0.7)
ax2.bar(x_pos + width / 2, means_poor, width, label="Bajo rendimiento", color="red", alpha=0.7)
ax2.set_xlabel("Estrategia")
ax2.set_ylabel("NDCG Promedio")
ax2.set_title("Rendimiento por Estrategia")
ax2.set_xticks(x_pos)
ax2.set_xticklabels(main_strategies, rotation=45)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análisis 5: Consenso entre estrategias
# Analizar si cuando múltiples estrategias coinciden en recomendaciones, mejora la calidad

print("=" * 80)
print("ANÁLISIS DE CONSENSO ENTRE ESTRATEGIAS")
print("=" * 80)

# Calcular correlación entre estrategias principales
print("\nCorrelación de NDCG entre estrategias:")
corr_matrix = df[[f"{s}_ndcg" for s in main_strategies]].corr()
print(corr_matrix.round(3))

# Visualización de matriz de correlación
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Correlación"},
    ax=ax,
    xticklabels=main_strategies,
    yticklabels=main_strategies,
)
ax.set_title("Correlación de Rendimiento entre Estrategias", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

# Análisis: ¿Usuarios donde estrategias están de acuerdo tienen mejor rendimiento?
# Calcular "discrepancia" entre estrategias como desviación estándar de sus NDCGs
df["strategy_std"] = df[[f"{s}_ndcg" for s in main_strategies]].std(axis=1)
df["strategy_mean"] = df[[f"{s}_ndcg" for s in main_strategies]].mean(axis=1)

# Segmentar por nivel de acuerdo (baja discrepancia = alto acuerdo)
df["high_consensus"] = df["strategy_std"] < df["strategy_std"].quantile(0.33)

print("\n" + "=" * 80)
print("ANÁLISIS DE CONSENSO VS RENDIMIENTO")
print("=" * 80)
print(
    f"\nUsuarios con alto consenso (baja discrepancia): {df['high_consensus'].sum()} ({df['high_consensus'].sum()/len(df)*100:.1f}%)"
)
print(
    f"Usuarios con bajo consenso (alta discrepancia): {(~df['high_consensus']).sum()} ({(~df['high_consensus']).sum()/len(df)*100:.1f}%)"
)

print("\nRendimiento promedio del híbrido:")
print(f"  Alto consenso: {df[df['high_consensus']]['hybrid_ndcg'].mean():.4f}")
print(f"  Bajo consenso: {df[~df['high_consensus']]['hybrid_ndcg'].mean():.4f}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análisis de Consenso entre Estrategias", fontsize=14, fontweight="bold")

# Distribución de discrepancia
ax1 = axes[0]
ax1.hist(df["strategy_std"], bins=50, alpha=0.7, color="steelblue", edgecolor="black")
ax1.axvline(
    df["strategy_std"].quantile(0.33), color="red", linestyle="--", label="Umbral alto consenso"
)
ax1.set_xlabel("Desviación Estándar de NDCG entre Estrategias")
ax1.set_ylabel("Frecuencia")
ax1.set_title("Distribución de Discrepancia entre Estrategias")
ax1.legend()
ax1.grid(alpha=0.3)

# Rendimiento por nivel de consenso
ax2 = axes[1]
consensus_data = [
    df[df["high_consensus"]]["hybrid_ndcg"].dropna(),
    df[~df["high_consensus"]]["hybrid_ndcg"].dropna(),
]
bp = ax2.boxplot(consensus_data, labels=["Alto consenso", "Bajo consenso"], patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("lightblue")
ax2.set_ylabel("NDCG@9 del Híbrido")
ax2.set_title("Rendimiento del Híbrido por Nivel de Consenso")
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Análisis 6: Modelado predictivo de mejor estrategia
# Intentar predecir qué estrategia funcionará mejor basándose en características del usuario

print("=" * 80)
print("MODELADO PREDICTIVO DE MEJOR ESTRATEGIA")
print("=" * 80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split


# Preparar datos para modelado
model_data = df[["holdout_size", "best_strategy"]].copy()
model_data = model_data[
    model_data["best_strategy"].isin(main_strategies)
]  # Solo estrategias principales

# Agregar características derivadas
model_data["log_holdout"] = np.log1p(model_data["holdout_size"])

# Crear variables dummy para segmentos
model_data["holdout_segment_encoded"] = pd.Categorical(model_data["holdout_size"]).codes

# Features para el modelo
features = ["holdout_size", "log_holdout"]
X = model_data[features]
y = model_data["best_strategy"]

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenar modelo simple
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

# Evaluar
y_pred = rf.predict(X_test)
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred, labels=main_strategies))

# Importancia de features
feature_importance = pd.DataFrame(
    {"feature": features, "importance": rf.feature_importances_}
).sort_values("importance", ascending=False)

print("\nImportancia de Features:")
print(feature_importance.to_string(index=False))

# Visualización de importancia
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feature_importance["feature"], feature_importance["importance"], color="steelblue")
ax.set_xlabel("Importancia")
ax.set_title("Importancia de Features para Predecir Mejor Estrategia", fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

# Análisis de errores: ¿Qué usuarios son más difíciles de predecir?
predictions_df = pd.DataFrame(
    {"holdout_size": X_test["holdout_size"], "actual": y_test, "predicted": y_pred}
)
predictions_df["correct"] = predictions_df["actual"] == predictions_df["predicted"]

print("\n" + "=" * 80)
print("ANÁLISIS DE ERRORES DE PREDICCIÓN")
print("=" * 80)
print(f"\nPrecisión general: {predictions_df['correct'].mean():.2%}")
print(
    f"\nTamaño promedio de holdout para predicciones correctas: {predictions_df[predictions_df['correct']]['holdout_size'].mean():.1f}"
)
print(
    f"Tamaño promedio de holdout para predicciones incorrectas: {predictions_df[~predictions_df['correct']]['holdout_size'].mean():.1f}"
)

In [ ]:
# Análisis 7: Simulación de Ensamble Simple
# Simular qué pasaría si combináramos estrategias con diferentes métodos

print("=" * 80)
print("SIMULACIÓN DE ENSAMBLE DE ESTRATEGIAS")
print("=" * 80)


# Método 1: Promedio simple de scores normalizados
def simulate_average_ensemble(row, strategies):
    """Simula ensamble por promedio simple"""
    scores = []
    for s in strategies:
        col = f"{s}_ndcg"
        if col in row.index and pd.notna(row[col]):
            scores.append(row[col])
    return np.mean(scores) if scores else 0.0


# Método 2: Máximo (mejor de todas)
def simulate_max_ensemble(row, strategies):
    """Simula ensamble por máximo"""
    scores = []
    for s in strategies:
        col = f"{s}_ndcg"
        if col in row.index and pd.notna(row[col]):
            scores.append(row[col])
    return np.max(scores) if scores else 0.0


# Método 3: Promedio ponderado (más peso a mejores estrategias)
def simulate_weighted_ensemble(row, strategies, weights):
    """Simula ensamble por promedio ponderado"""
    score = 0.0
    total_weight = 0.0
    for s, w in zip(strategies, weights, strict=False):
        col = f"{s}_ndcg"
        if col in row.index and pd.notna(row[col]):
            score += row[col] * w
            total_weight += w
    return score / total_weight if total_weight > 0 else 0.0


# Aplicar simulaciones
df["ensemble_avg"] = df.apply(lambda row: simulate_average_ensemble(row, main_strategies), axis=1)
df["ensemble_max"] = df.apply(lambda row: simulate_max_ensemble(row, main_strategies), axis=1)

# Pesos basados en rendimiento promedio de cada estrategia
strategy_weights = []
for s in main_strategies:
    col = f"{s}_ndcg"
    if col in df.columns:
        mean_score = df[col].mean()
        strategy_weights.append(mean_score)
    else:
        strategy_weights.append(0.0)

# Normalizar pesos
total_weight = sum(strategy_weights)
strategy_weights = [
    w / total_weight if total_weight > 0 else 1.0 / len(main_strategies) for w in strategy_weights
]

df["ensemble_weighted"] = df.apply(
    lambda row: simulate_weighted_ensemble(row, main_strategies, strategy_weights), axis=1
)

# Comparar métodos de ensamble
ensemble_comparison = pd.DataFrame(
    {
        "Método": ["Híbrido Actual", "Promedio Simple", "Máximo", "Promedio Ponderado"],
        "NDCG Promedio": [
            df["hybrid_ndcg"].mean(),
            df["ensemble_avg"].mean(),
            df["ensemble_max"].mean(),
            df["ensemble_weighted"].mean(),
        ],
        "NDCG Mediano": [
            df["hybrid_ndcg"].median(),
            df["ensemble_avg"].median(),
            df["ensemble_max"].median(),
            df["ensemble_weighted"].median(),
        ],
    }
)

print("\nComparación de Métodos de Ensamble:")
print(ensemble_comparison.round(4).to_string(index=False))

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparación de Métodos de Ensamble", fontsize=14, fontweight="bold")

# Boxplot comparativo
ax1 = axes[0]
ensemble_data = [
    df["hybrid_ndcg"].dropna(),
    df["ensemble_avg"].dropna(),
    df["ensemble_max"].dropna(),
    df["ensemble_weighted"].dropna(),
]
bp = ax1.boxplot(
    ensemble_data, labels=["Híbrido Actual", "Promedio", "Máximo", "Ponderado"], patch_artist=True
)
for patch in bp["boxes"]:
    patch.set_facecolor("lightblue")
ax1.set_ylabel("NDCG@9")
ax1.set_title("Distribución de Rendimiento")
ax1.tick_params(axis="x", rotation=45)
ax1.grid(axis="y", alpha=0.3)

# Barras de promedio
ax2 = axes[1]
ax2.bar(
    ensemble_comparison["Método"],
    ensemble_comparison["NDCG Promedio"],
    color="steelblue",
    alpha=0.7,
)
ax2.set_ylabel("NDCG Promedio")
ax2.set_title("Rendimiento Promedio por Método")
ax2.tick_params(axis="x", rotation=45)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Análisis: ¿En qué usuarios el ensamble mejora más?
df["ensemble_improvement"] = df["ensemble_weighted"] - df["hybrid_ndcg"]
improved_users = df[df["ensemble_improvement"] > 0.01]  # Mejora significativa

print("\n" + "=" * 80)
print("POTENCIAL DE MEJORA CON ENSAMBLE")
print("=" * 80)
print(
    f"\nUsuarios que mejorarían con ensamble ponderado: {len(improved_users)} ({len(improved_users)/len(df)*100:.1f}%)"
)
print(f"Mejora promedio: {improved_users['ensemble_improvement'].mean():.4f}")
print(f"Mejora máxima: {df['ensemble_improvement'].max():.4f}")

if len(improved_users) > 0:
    print("\nCaracterísticas de usuarios que más mejoran:")
    print(f"  Tamaño promedio de holdout: {improved_users['holdout_size'].mean():.1f}")
    print(f"  Mejora promedio: {improved_users['ensemble_improvement'].mean():.4f}")

In [ ]:
# Análisis 8: Estrategias complementarias por segmento
# Identificar qué estrategias funcionan mejor juntas en diferentes segmentos

print("=" * 80)
print("ANÁLISIS DE ESTRATEGIAS COMPLEMENTARIAS POR SEGMENTO")
print("=" * 80)

# Para cada segmento, identificar qué estrategias son mejores
segment_strategy_analysis = []

for segment in df["holdout_segment"].cat.categories:
    subset = df[df["holdout_segment"] == segment]
    if len(subset) > 0:
        row = {"segment": segment, "user_count": len(subset)}

        # Mejor estrategia promedio
        best_strategy = None
        best_mean = -1
        for strategy in main_strategies:
            col = f"{strategy}_ndcg"
            if col in subset.columns:
                mean_ndcg = subset[col].mean()
                row[f"{strategy}_mean"] = mean_ndcg
                if mean_ndcg > best_mean:
                    best_mean = mean_ndcg
                    best_strategy = strategy

        row["best_strategy"] = best_strategy

        # Estrategias que funcionan bien (percentil 75)
        good_strategies = []
        for strategy in main_strategies:
            col = f"{strategy}_ndcg"
            if col in subset.columns:
                threshold = subset[col].quantile(0.75)
                if subset[col].mean() >= threshold:
                    good_strategies.append(strategy)

        row["good_strategies"] = ", ".join(good_strategies) if good_strategies else "Ninguna"
        segment_strategy_analysis.append(row)

segment_strategy_df = pd.DataFrame(segment_strategy_analysis)
print("\nMejor estrategia por segmento:")
print(
    segment_strategy_df[["segment", "user_count", "best_strategy", "good_strategies"]].to_string(
        index=False
    )
)

# Visualización: Heatmap de rendimiento por segmento y estrategia
pivot_data = []
for segment in df["holdout_segment"].cat.categories:
    subset = df[df["holdout_segment"] == segment]
    row = {"segment": segment}
    for strategy in main_strategies:
        col = f"{strategy}_ndcg"
        if col in subset.columns:
            row[strategy] = subset[col].mean()
        else:
            row[strategy] = 0.0
    pivot_data.append(row)

pivot_df = pd.DataFrame(pivot_data).set_index("segment")

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    pivot_df,
    annot=True,
    fmt=".3f",
    cmap="YlOrRd",
    cbar_kws={"label": "NDCG Promedio"},
    ax=ax,
    linewidths=0.5,
)
ax.set_title(
    "Rendimiento de Estrategias por Segmento de Holdout", fontsize=14, fontweight="bold", pad=20
)
ax.set_xlabel("Estrategia", fontweight="bold")
ax.set_ylabel("Segmento de Holdout", fontweight="bold")
plt.tight_layout()
plt.show()

## Conclusiones y Recomendaciones para Sistema Híbrido Mejorado

### Insights Clave:

1. **Complementariedad de Estrategias**:
   - [Basado en análisis de complementariedad]
   - Las estrategias tienen diferentes fortalezas según el usuario

2. **Segmentación de Usuarios**:
   - [Basado en análisis por tamaño de holdout]
   - Diferentes estrategias funcionan mejor para diferentes segmentos

3. **Potencial de Ensamble**:
   - [Basado en simulación de ensamble]
   - Combinar estrategias puede mejorar resultados en [X]% de usuarios

4. **Casos Difíciles**:
   - [Basado en análisis de bajo rendimiento]
   - Usuarios con [características] requieren tratamiento especial

5. **Consenso entre Estrategias**:
   - [Basado en análisis de consenso]
   - Cuando múltiples estrategias coinciden, la calidad mejora

### Recomendaciones para Sistema Híbrido:

#### Opción 1: Modelado Adaptativo
- **Enfoque**: Predecir la mejor estrategia según características del usuario
- **Ventajas**: Simple, interpretable, fácil de implementar
- **Cuándo usar**: Si hay patrones claros de qué estrategia funciona mejor

#### Opción 2: Ensamble Inteligente
- **Enfoque**: Combinar múltiples estrategias con pesos adaptativos
- **Ventajas**: Aprovecha fortalezas de múltiples estrategias
- **Cuándo usar**: Si múltiples estrategias funcionan bien simultáneamente

#### Opción 3: Sistema Híbrido por Segmentos
- **Enfoque**: Reglas diferentes según segmento de usuario
- **Ventajas**: Balance entre simplicidad y efectividad
- **Cuándo usar**: Si hay segmentos claros con necesidades diferentes



In [ ]:
# Resumen ejecutivo: Guardar insights clave para referencia futura

insights_summary = {
    "total_users": len(df),
    "best_strategy_distribution": df["best_strategy"].value_counts().to_dict(),
    "ensemble_potential": {
        "users_with_multiple_good_strategies": int((df["good_strategies_count"] >= 2).sum()),
        "percentage": float((df["good_strategies_count"] >= 2).sum() / len(df) * 100),
        "avg_improvement_with_weighted_ensemble": float(df["ensemble_improvement"].mean()),
    },
    "strategy_performance_by_segment": segment_strategy_df[["segment", "best_strategy"]].to_dict(
        "records"
    ),
    "consensus_analysis": {
        "high_consensus_users": int(df["high_consensus"].sum()),
        "high_consensus_hybrid_ndcg": float(df[df["high_consensus"]]["hybrid_ndcg"].mean()),
        "low_consensus_hybrid_ndcg": float(df[~df["high_consensus"]]["hybrid_ndcg"].mean()),
    },
    "poor_performers": {
        "count": int(len(poor_performers)),
        "percentage": float(len(poor_performers) / len(df) * 100),
        "avg_holdout_size": float(poor_performers["holdout_size"].mean()),
    },
}

import json


with open("../insights_hibrido.json", "w") as f:
    json.dump(insights_summary, f, indent=2, default=str)

print("✅ Insights guardados en: insights_hibrido.json")
print("\nResumen de Insights Clave:")
print(f"  - Total usuarios analizados: {insights_summary['total_users']}")
print(
    f"  - Usuarios con potencial de ensamble: {insights_summary['ensemble_potential']['users_with_multiple_good_strategies']} ({insights_summary['ensemble_potential']['percentage']:.1f}%)"
)
print(
    f"  - Mejora promedio con ensamble ponderado: {insights_summary['ensemble_potential']['avg_improvement_with_weighted_ensemble']:.4f}"
)
print(
    f"  - Usuarios con bajo rendimiento: {insights_summary['poor_performers']['count']} ({insights_summary['poor_performers']['percentage']:.1f}%)"
)